In [0]:
WITH params AS (
  SELECT 30 AS p_ml_window_days
),
runs_scoped AS (
  SELECT
    CAST(r.run_id AS STRING) AS run_id,
    CAST(r.workspace_id AS STRING) AS workspace_id,
    CAST(r.experiment_id AS STRING) AS experiment_id,
    r.run_name,
    r.status,
    r.created_by,
    r.start_time AS run_start_time_utc,
    r.end_time AS run_end_time_utc
  FROM system.mlflow.runs_latest r
  CROSS JOIN params p
  WHERE r.delete_time IS NULL
    AND r.start_time >= timestampadd(DAY, -p.p_ml_window_days, current_timestamp())
),
metrics_scoped AS (
  SELECT
    CAST(h.run_id AS STRING) AS run_id,
    lower(h.metric_name) AS metric_name_lc,
    CAST(h.metric_value AS DOUBLE) AS metric_value,
    h.metric_time,
    h.metric_step
  FROM system.mlflow.run_metrics_history h
  CROSS JOIN params p
  INNER JOIN runs_scoped r
    ON CAST(h.run_id AS STRING) = r.run_id
  WHERE h.metric_time >= timestampadd(DAY, -p.p_ml_window_days, current_timestamp())
),
metric_classified AS (
  SELECT
    run_id,
    metric_name_lc,
    metric_value,
    metric_time,
    metric_step,
    CASE
      WHEN metric_name_lc = 'training_score' THEN 'train_score'
      WHEN metric_name_lc = 'val_score' THEN 'val_score'
      WHEN metric_name_lc = 'test_score' THEN 'test_score'

      WHEN metric_name_lc IN ('training_r2_score', 'r2_score_x_train') THEN 'train_r2'
      WHEN metric_name_lc IN ('val_r2_score', 'r2_score_x_val') THEN 'val_r2'
      WHEN metric_name_lc IN ('test_r2_score', 'r2_score_x_test') THEN 'test_r2'

      WHEN metric_name_lc = 'training_root_mean_squared_error' THEN 'train_rmse'
      WHEN metric_name_lc = 'val_root_mean_squared_error' THEN 'val_rmse'
      WHEN metric_name_lc = 'test_root_mean_squared_error' THEN 'test_rmse'

      WHEN metric_name_lc IN ('training_mean_squared_error', 'mean_squared_error_x_train') THEN 'train_mse'
      WHEN metric_name_lc IN ('val_mean_squared_error', 'mean_squared_error_x_val') THEN 'val_mse'
      WHEN metric_name_lc IN ('test_mean_squared_error', 'mean_squared_error_x_test') THEN 'test_mse'

      WHEN metric_name_lc = 'training_mean_absolute_error' THEN 'train_mae'
      WHEN metric_name_lc = 'val_mean_absolute_error' THEN 'val_mae'
      WHEN metric_name_lc = 'test_mean_absolute_error' THEN 'test_mae'
      ELSE NULL
    END AS metric_bucket
  FROM metrics_scoped
),
metric_latest_ranked AS (
  SELECT
    run_id,
    metric_bucket,
    metric_value,
    ROW_NUMBER() OVER (
      PARTITION BY run_id, metric_bucket
      ORDER BY metric_time DESC, metric_step DESC, metric_value DESC
    ) AS rn
  FROM metric_classified
  WHERE metric_bucket IS NOT NULL
),
metric_latest AS (
  SELECT
    run_id,
    MAX(CASE WHEN metric_bucket = 'train_score' THEN metric_value END) AS training_score_latest,
    MAX(CASE WHEN metric_bucket = 'val_score' THEN metric_value END) AS val_score_latest,
    MAX(CASE WHEN metric_bucket = 'test_score' THEN metric_value END) AS test_score_latest,

    MAX(CASE WHEN metric_bucket = 'train_r2' THEN metric_value END) AS training_r2_latest,
    MAX(CASE WHEN metric_bucket = 'val_r2' THEN metric_value END) AS val_r2_latest,
    MAX(CASE WHEN metric_bucket = 'test_r2' THEN metric_value END) AS test_r2_latest,

    MAX(CASE WHEN metric_bucket = 'train_rmse' THEN metric_value END) AS training_rmse_latest,
    MAX(CASE WHEN metric_bucket = 'val_rmse' THEN metric_value END) AS val_rmse_latest,
    MAX(CASE WHEN metric_bucket = 'test_rmse' THEN metric_value END) AS test_rmse_latest,

    MAX(CASE WHEN metric_bucket = 'train_mse' THEN metric_value END) AS training_mse_latest,
    MAX(CASE WHEN metric_bucket = 'val_mse' THEN metric_value END) AS val_mse_latest,
    MAX(CASE WHEN metric_bucket = 'test_mse' THEN metric_value END) AS test_mse_latest,

    MAX(CASE WHEN metric_bucket = 'train_mae' THEN metric_value END) AS training_mae_latest,
    MAX(CASE WHEN metric_bucket = 'val_mae' THEN metric_value END) AS val_mae_latest,
    MAX(CASE WHEN metric_bucket = 'test_mae' THEN metric_value END) AS test_mae_latest
  FROM metric_latest_ranked
  WHERE rn = 1
  GROUP BY run_id
),
metric_best AS (
  SELECT
    run_id,
    MAX(CASE WHEN metric_bucket = 'train_score' THEN metric_value END) AS training_score_max,
    MAX(CASE WHEN metric_bucket = 'val_score' THEN metric_value END) AS val_score_max,
    MAX(CASE WHEN metric_bucket = 'test_score' THEN metric_value END) AS test_score_max,

    MAX(CASE WHEN metric_bucket = 'train_r2' THEN metric_value END) AS training_r2_max,
    MAX(CASE WHEN metric_bucket = 'val_r2' THEN metric_value END) AS val_r2_max,
    MAX(CASE WHEN metric_bucket = 'test_r2' THEN metric_value END) AS test_r2_max,

    MIN(CASE WHEN metric_bucket = 'train_rmse' THEN metric_value END) AS training_rmse_min,
    MIN(CASE WHEN metric_bucket = 'val_rmse' THEN metric_value END) AS val_rmse_min,
    MIN(CASE WHEN metric_bucket = 'test_rmse' THEN metric_value END) AS test_rmse_min,

    MIN(CASE WHEN metric_bucket = 'train_mse' THEN metric_value END) AS training_mse_min,
    MIN(CASE WHEN metric_bucket = 'val_mse' THEN metric_value END) AS val_mse_min,
    MIN(CASE WHEN metric_bucket = 'test_mse' THEN metric_value END) AS test_mse_min,

    MIN(CASE WHEN metric_bucket = 'train_mae' THEN metric_value END) AS training_mae_min,
    MIN(CASE WHEN metric_bucket = 'val_mae' THEN metric_value END) AS val_mae_min,
    MIN(CASE WHEN metric_bucket = 'test_mae' THEN metric_value END) AS test_mae_min
  FROM metric_classified
  WHERE metric_bucket IS NOT NULL
  GROUP BY run_id
),
training_metric_candidates AS (
  SELECT
    run_id,
    metric_name_lc AS training_data_count_key,
    metric_value AS training_data_count,
    metric_time AS training_data_count_observed_at_utc,
    CASE
      WHEN metric_name_lc IN (
        'training_example_count',
        'train_example_count',
        'training_sample_count',
        'train_sample_count',
        'training_row_count',
        'train_row_count',
        'training_rows',
        'train_rows',
        'dataset_row_count',
        'dataset_rows',
        'num_train_rows',
        'num_train_samples'
      ) THEN 220
      WHEN metric_name_lc RLIKE '(train|training|dataset).*(example|sample|row|record).*(count|num|size)' THEN 170
      WHEN metric_name_lc RLIKE '(example|sample|row|record).*(count|num|size).*(train|training|dataset)' THEN 160
      WHEN metric_name_lc RLIKE '(train|training|dataset).*(example|sample|row|record)' THEN 130
      ELSE 70
    END AS candidate_score
  FROM metrics_scoped
  WHERE metric_value IS NOT NULL
    AND NOT isnan(metric_value)
    AND metric_value > 0
    AND metric_value < 1000000000000000
    AND metric_name_lc RLIKE '(train|training|dataset|example|examples|sample|samples|row|rows|record|records)'
),
training_metric_best AS (
  SELECT
    run_id,
    training_data_count,
    training_data_count_key,
    training_data_count_observed_at_utc
  FROM (
    SELECT
      run_id,
      training_data_count,
      training_data_count_key,
      training_data_count_observed_at_utc,
      candidate_score,
      ROW_NUMBER() OVER (
        PARTITION BY run_id
        ORDER BY candidate_score DESC, training_data_count_observed_at_utc DESC, training_data_count DESC
      ) AS rn
    FROM training_metric_candidates
    WHERE training_data_count IS NOT NULL
  ) ranked
  WHERE rn = 1
),
metric_key_profile AS (
  SELECT
    run_id,
    concat_ws(', ', slice(array_sort(collect_set(metric_name_lc)), 1, 50)) AS metric_keys_detected
  FROM metrics_scoped
  GROUP BY run_id
)
SELECT
  r.workspace_id,
  r.experiment_id,
  r.run_id,
  r.run_name,
  r.status,
  r.created_by,
  r.run_start_time_utc,
  from_utc_timestamp(r.run_start_time_utc, 'Asia/Seoul') AS run_start_time_kst,
  r.run_end_time_utc,
  from_utc_timestamp(r.run_end_time_utc, 'Asia/Seoul') AS run_end_time_kst,
  CASE
    WHEN r.run_end_time_utc IS NOT NULL AND r.run_start_time_utc IS NOT NULL
    THEN unix_timestamp(r.run_end_time_utc) - unix_timestamp(r.run_start_time_utc)
    ELSE NULL
  END AS run_duration_seconds,

  ml.training_score_latest,
  mb.training_score_max,
  ml.val_score_latest,
  mb.val_score_max,
  ml.test_score_latest,
  mb.test_score_max,

  ml.training_r2_latest,
  mb.training_r2_max,
  ml.val_r2_latest,
  mb.val_r2_max,
  ml.test_r2_latest,
  mb.test_r2_max,

  ml.training_rmse_latest,
  mb.training_rmse_min,
  ml.val_rmse_latest,
  mb.val_rmse_min,
  ml.test_rmse_latest,
  mb.test_rmse_min,

  ml.training_mse_latest,
  mb.training_mse_min,
  ml.val_mse_latest,
  mb.val_mse_min,
  ml.test_mse_latest,
  mb.test_mse_min,

  ml.training_mae_latest,
  mb.training_mae_min,
  ml.val_mae_latest,
  mb.val_mae_min,
  ml.test_mae_latest,
  mb.test_mae_min,

  'regression' AS model_family,
  CASE
    WHEN ml.test_r2_latest IS NOT NULL THEN 'test_r2_score'
    WHEN ml.val_r2_latest IS NOT NULL THEN 'val_r2_score'
    WHEN ml.training_r2_latest IS NOT NULL THEN 'training_r2_score'
    WHEN ml.test_score_latest IS NOT NULL THEN 'test_score'
    WHEN ml.val_score_latest IS NOT NULL THEN 'val_score'
    WHEN ml.training_score_latest IS NOT NULL THEN 'training_score'
    WHEN ml.test_rmse_latest IS NOT NULL THEN 'test_root_mean_squared_error'
    WHEN ml.val_rmse_latest IS NOT NULL THEN 'val_root_mean_squared_error'
    WHEN ml.training_rmse_latest IS NOT NULL THEN 'training_root_mean_squared_error'
    ELSE NULL
  END AS primary_metric_name,
  COALESCE(
    ml.test_r2_latest,
    ml.val_r2_latest,
    ml.training_r2_latest,
    ml.test_score_latest,
    ml.val_score_latest,
    ml.training_score_latest,
    ml.test_rmse_latest,
    ml.val_rmse_latest,
    ml.training_rmse_latest
  ) AS primary_metric_latest,

  tb.training_data_count,
  CASE
    WHEN tb.training_data_count IS NOT NULL THEN 'metrics_history'
    ELSE 'not_found'
  END AS training_data_count_source,
  tb.training_data_count_key,
  tb.training_data_count_observed_at_utc,
  from_utc_timestamp(tb.training_data_count_observed_at_utc, 'Asia/Seoul') AS training_data_count_observed_at_kst,

  mkp.metric_keys_detected,
  current_timestamp() AS generated_at_utc,
  from_utc_timestamp(current_timestamp(), 'Asia/Seoul') AS generated_at_kst
FROM runs_scoped r
LEFT JOIN metric_latest ml
  ON r.run_id = ml.run_id
LEFT JOIN metric_best mb
  ON r.run_id = mb.run_id
LEFT JOIN training_metric_best tb
  ON r.run_id = tb.run_id
LEFT JOIN metric_key_profile mkp
  ON r.run_id = mkp.run_id
ORDER BY COALESCE(r.run_end_time_utc, r.run_start_time_utc) DESC, r.run_id DESC;